# LlmEdith : PyTorch sur GPU Colab
*Exécution → Modifier le type d'exécution → GPU (T4)*. Secrets 🔑 : `HF_TOKEN`, `GITHUB_TOKEN`.
Idéal pour les ablations `tiny`, le SFT et pour discuter avec le modèle.

In [ ]:
import os, subprocess
from google.colab import userdata
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
try:
    GH = userdata.get('GITHUB_TOKEN')  # seulement si le dépôt GitHub est privé
except Exception:
    GH = None
os.environ['LLMEDITH_HUB_REPO'] = 'HarpePluie/llmedith-checkpoints'
repo = 'https://github.com/HarpeLm/LlmEdith.git'.replace('https://', f'https://{GH}@') if GH else 'https://github.com/HarpeLm/LlmEdith.git'
if not os.path.exists('LlmEdith'):
    subprocess.run(['git', 'clone', '-q', repo, 'LlmEdith'], check=True)
else:
    subprocess.run(['git', '-C', 'LlmEdith', 'pull', '-q'], check=True)
os.chdir('LlmEdith')

In [ ]:
!pip install -q -e '.[eval]'

## Ablation `tiny` (Muon vs AdamW)

In [ ]:
!python -m llmedith.torch_impl.train --config configs/tiny.yaml --resume --max_minutes 170 \
    --override data.hub_repo=HarpePluie/llmedith-data
# puis : --override name=tiny_adamw train.optimizer=adamw  pour comparer

## Discuter avec le modèle aligné

In [ ]:
from llmedith import hub
hub.pull_folder('HarpePluie/llmedith-checkpoints', 'dpo/latest', 'runs')
!python -m convert.to_hf --ckpt runs/dpo/latest --out exports/llmedith-chat --chat
!python -m posttrain.chat exports/llmedith-chat